# Browse seed-split eval responses

Same `get_responses` / `get_scores` interface as the FT browse
notebook, but indexed by an extra `seed` dimension. Each
`scores_<base_model>_seed-<seed>.json` in this directory is loaded
into `SCORES[base_model][seed]`.

- **model** -- the base LLM (`meta-llama-Llama-3.1-8B-Instruct`, ...).
- **seed** -- the finetune seed string (`'default'`, `'2'`, `'3'`,
  `'5'`, ...). `'default'` is the run with no explicit seed.
- **pole** -- the propensity the checkpoint was finetuned toward
  (`agreeableness-plus`, ...). `'base'` is the unfinetuned model and
  appears in every seed.
- **eval** -- which propensity is being measured.

On-diagonal cells (`X-plus`/`X-minus` x eval `X`) are direct
elicitations; off-diagonal cells are cross-elicitations.

Re-running `summarize_seeds.py` does **not** overwrite this notebook --
delete it first if you want the fresh scaffold.


In [ ]:
import json, re
from pathlib import Path

_here = Path('.').resolve()
SEEDS_DIR = _here if _here.name == 'seeds' else _here / 'results' / 'seeds'
EVAL_ROOT = (SEEDS_DIR.parent.parent / 'eval_results' / 'finetuning').resolve()

_NAME_RE = re.compile(r'^scores_(?P<model>.+)_seed-(?P<seed>.+)\.json$')
SCORES = {}
for p in sorted(SEEDS_DIR.glob('scores_*_seed-*.json')):
    m = _NAME_RE.match(p.name)
    if not m:
        continue
    doc = json.loads(p.read_text())
    if doc.get('n_cells', 0) == 0:
        continue
    SCORES.setdefault(doc['base_model'], {})[doc['seed']] = doc

print('Loaded scores:')
for model, by_seed in SCORES.items():
    seeds = sorted(by_seed, key=lambda s: (s != 'default', s))
    cells = sum(d['n_cells'] for d in by_seed.values())
    print(f'  {model}  seeds={seeds}  ({cells} cells total)')
print()
print('Use get_responses(model, seed, pole, eval) and get_scores(model, seed, pole, eval).')


In [ ]:
def _cell(model, seed, pole, eval_propensity):
    if model not in SCORES:
        raise KeyError(f'unknown model {model!r}; loaded: {sorted(SCORES)}')
    by_seed = SCORES[model]
    if seed not in by_seed:
        raise KeyError(
            f'no seed {seed!r} for model {model!r}; available: {sorted(by_seed)}'
        )
    cells = by_seed[seed]['cells']
    if pole not in cells:
        raise KeyError(
            f'unknown pole {pole!r} for {model!r} seed={seed!r}; '
            f'available: {sorted(cells)}'
        )
    if eval_propensity not in cells[pole]:
        raise KeyError(
            f'no eval {eval_propensity!r} for pole {pole!r} seed={seed!r}; '
            f'available: {sorted(cells[pole])}'
        )
    return cells[pole][eval_propensity]


def _iter_rows(model, seed, pole, eval_propensity):
    cell = _cell(model, seed, pole, eval_propensity)
    rows_path = EVAL_ROOT / cell['meta']['dirname'] / 'rows.jsonl'
    if not rows_path.exists():
        raise FileNotFoundError(f'missing rows.jsonl: {rows_path}')
    with rows_path.open() as f:
        for line in f:
            yield json.loads(line)


def get_responses(model, seed, pole, eval_propensity):
    """Conversations from `model`'s seed=`seed` `pole` checkpoint on `eval_propensity`.

    Returns [{'question', 'answer'}] in rows.jsonl order.
    """
    return [
        {'question': r.get('question'), 'answer': r.get('answer')}
        for r in _iter_rows(model, seed, pole, eval_propensity)
    ]


def get_scores(model, seed, pole, eval_propensity):
    """Per-conversation judge scores, in the same order as get_responses(...)."""
    return [r.get('score') for r in _iter_rows(model, seed, pole, eval_propensity)]


## Example: compare seeds for one (pole, eval) cell

In [ ]:
# Pick the first model/pole/eval triple that exists and print one
# conversation per available seed for comparison.
if SCORES:
    model = next(iter(SCORES))
    by_seed = SCORES[model]
    any_seed = next(iter(by_seed))
    cells = by_seed[any_seed]['cells']
    pole = 'neuroticism-plus' if 'neuroticism-plus' in cells else next(iter(cells))
    eval_p = 'neuroticism' if 'neuroticism' in cells[pole] else next(iter(cells[pole]))

    seeds = sorted(by_seed, key=lambda s: (s != 'default', s))
    print(f'{model} | {pole} | {eval_p}')
    for s in seeds:
        try:
            convos = get_responses(model, s, pole, eval_p)
            scores = get_scores(model, s, pole, eval_p)
        except KeyError as exc:
            print(f'  seed={s}: <{exc.args[0]}>')
            continue
        if not convos:
            print(f'  seed={s}: (no conversations)')
            continue
        print(f'  ---- seed={s}  (n={len(convos)}) ----')
        print('  Q:', (convos[0]["question"] or '')[:200])
        print('  A:', (convos[0]["answer"] or '')[:200])
        print('  score:', scores[0])
